In [1]:
import climate_learn as cl
import numpy as np
import xarray as xr

### ERA5 dataset
ERA5 is a reanalysis dataset maintained by the European Center for Medium-Range Weather Forecasting (ECMWF). In its raw format, ERA5 contains hourly data from 1979 to the current time on a grid with cells of width and height 0.25 degree of the Earth, with different climate variables at 37 different pressure levels plus the planet’s surface. This corresponds to nearly 400,000 data samples, each a matrix of shape 721*1440. Since this is too big for most deep learning models, ClimateLearn supports downloading a smaller, pre-processed version of ERA5 data from WeatherBench.




#### downloaded 5.626 degree: 
##### 13-pressure-level
- <span style="color:red">geopotential</span> (all levels: 50,  100,  150,  200,  250,  300,  400, <span style="color:red">500</span> ,  600, 700, 850,  925, 1000 hPa)
- <span style="color:red">temperature</span> (all levels: 50,  100,  150,  200,  250,  300,  400,  500,  600, 700,  500, <span style="color:red">850</span>,  925, 1000 hPa)
- relative_humidity (all levels: 50,  100,  150,  200,  250,  300,  400,  500,  600, 700,  850,  925, 1000 hPa)
- specific_humidity (all levels: 50,  100,  150,  200,  250,  300,  400,  500,  600, 700,  850,  925, 1000 hPa)
- u_component_of_wind (all levels: 50,  100,  150,  200,  250,  300,  400,  500,  600, 700,  850,  925, 1000 hPa)
- v_component_of_wind (all levels: 50,  100,  150,  200,  250,  300,  400,  500,  600, 700,  850,  925, 1000 hPa)

##### single-surface-level
- <span style="color:red">2m_temperature</span> 
- <span style="color:red">10m_u_component_of_wind</span> 
- <span style="color:red">10m_v_component_of_wind</span> 
- total_precipitation
- total_cloud_cover
- toa_incident_solar_radiation
  
We mark the data to download in <span style="color:red">red</span>.



gsutil -m cp -r gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr/temperature .

In [ ]:
# for the weatherbench data (codicast default stuff)
root_directory = "/mnt/data/sonia/weatherbench-5.625"  

import xarray as xr
import pandas as pd
import os

varnames=['10m_u_component_of_wind', '10m_v_component_of_wind',         '2m_temperature']
for varname in varnames:
    os.makedirs(f'{root_directory}/{varname}', exist_ok=True)


ds = xr.open_zarr(f'{root_directory}/zarr', consolidated=False,)

start = '1959-01-01 00:00:00'
correct_time = pd.date_range(start=pd.to_datetime(start), periods=len(ds.time), freq='6h')
ds = ds.assign_coords(time=correct_time)
min_year = 1959
max_year = int(str(ds['time'].max().values)[:4]) + 1 # exlusive
for varname in varnames:
    for year in range(min_year, max_year):
        ds_year = ds.sel(time=str(year))[varname]
        ds_year.to_netcdf(f'{root_directory}/{varname}/{varname}_{year}_5.625deg.nc')



In [ ]:
# for the era5 data (my stuff)
import os 

root_directory = '/mnt/data/sonia/cyclone/0.25'
target_directory = '/mnt/data/sonia/codicast-data/multivar'
dirnames=['slp', 'wind_500hpa', 'wind_500hpa', 'temperature', 'humidity']
shortnames = ['slp']#, 'u', 'v', 't', 'q']

os.makedirs(os.path.join(target_directory, 'temporary'), exist_ok=True)

for var, short in zip(dirnames, shortnames):
    os.makedirs(os.path.join(target_directory, var), exist_ok=True)
    for year in range(1940, 1941):
        if 'wind' in var:
            fname = f'wind.{year}.nc'
        else:
            fname = f'{var}.{year}.nc'
            
        ds = xr.open_dataset(os.path.join(root_directory, var, fname))


### Create train / val / test

In [ ]:
def select_merge_data(var_list, year_start, year_end, data_folder, resolution, lat, long):
    directory_paths = var_list
    concat_years = []
    counts = 0
    years = []
    
    for year in range(year_start, year_end+1):
        years.append(str(year))

    for year in years:
        print('>>>', year, '<<<')
        for directory_path in directory_paths:
            # # Open the NetCDF file using xarray
            # ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + '_' + year + '_' + str(resolution) + 'deg.nc')
    
            # # Select every 6th sample
            # ds = ds.isel(time=slice(None, None, 6))
        
            # =========== pressure-level =============  
            if directory_path == 'geopotential_500':
                ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + 'hPa_' + year + '_' + str(resolution) + 'deg.nc')
                ds = ds.isel(time=slice(None, None, 6))
                geopotential = ds['z'].values
                geopotential = geopotential.reshape((-1, 1, lat, long))
                print('geopotential_500:', geopotential.shape)
                
            if directory_path == 'temperature_850':
                ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + 'hPa_' + year + '_' + str(resolution) + 'deg.nc')
                ds = ds.isel(time=slice(None, None, 6))
                temperature = ds['t'].values
                temperature = temperature.reshape((-1, 1, lat, long))
                print('temperature_850:', temperature.shape)
        
            # ======================= surface variable ======================
            if directory_path == '2m_temperature':
                ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + '_' + year + '_' + str(resolution) + 'deg.nc')
                # ds = ds.isel(time=slice(None, None, 6))
                t2m_temperature = ds['2m_temperature'].values
                t2m_temperature = t2m_temperature.reshape((-1, 1, lat, long))
                print('2m_temperature:', t2m_temperature.shape)
        
            if directory_path == '10m_u_component_of_wind':
                ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + '_' + year + '_' + str(resolution) + 'deg.nc')
                # ds = ds.isel(time=slice(None, None, 6))
                u10m = ds['10m_u_component_of_wind'].values
                u10m = u10m.reshape((-1, 1, lat, long))
                print('10m_u_component_of_wind:', u10m.shape)
        
            if directory_path == '10m_v_component_of_wind': 
                ds = xr.open_dataset(data_folder + '/' + directory_path + '/' + directory_path + '_' + year + '_' + str(resolution) + 'deg.nc')
                print(ds)
                # ds = ds.isel(time=slice(None, None, 6))
                v10m = ds['10m_v_component_of_wind'].values
                v10m = v10m.reshape((-1, 1,lat, long))
                print('10m_v_component_of_wind:', v10m.shape)
        
        # concatenate one year
        for ds in [geopotential, temperature, t2m_temperature, u10m, v10m]:
            print(ds.shape, np.isnan(ds).sum())
        concat_one_year = np.concatenate([geopotential, temperature, t2m_temperature, u10m, v10m], axis=1)        
        print("concat_one_year.shape:", concat_one_year.shape)
    
        concat_years.append(concat_one_year)
    
        counts += concat_one_year.shape[0]

    concat_years = np.concatenate(concat_years, axis=0)
    
    print("concat_years.shape:", concat_years.shape)
    
    print("total time points:", counts)

    print(">>> saving data <<<") 
    np.save(data_folder + '/concat_' + str(year_start) + '_' + str(year_end) + '_' + str(resolution) + '_' + str(concat_years.shape[1]) + 'var.npy', concat_years)
    

    print(">>> saved data <<<")

### Test set

In [26]:
var_list = ['geopotential_500', 'temperature_850', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind']

year_start, year_end = 2017, 2018

resolution = 5.625 

data_folder = '/mnt/data/sonia/weatherbench-5.625'

lat, long = 32, 64

In [27]:
select_merge_data(var_list, year_start, year_end, data_folder, resolution, lat, long)

>>> 2017 <<<
geopotential_500: (1460, 1, 32, 64)
temperature_850: (1460, 1, 32, 64)
2m_temperature: (1460, 1, 32, 64)
10m_u_component_of_wind: (1460, 1, 32, 64)
<xarray.Dataset> Size: 12MB
Dimensions:                  (time: 1460, latitude: 64, longitude: 32)
Coordinates:
  * time                     (time) datetime64[ns] 12kB 2017-01-01 ... 2017-1...
Dimensions without coordinates: latitude, longitude
Data variables:
    10m_v_component_of_wind  (time, latitude, longitude) float32 12MB ...
10m_v_component_of_wind: (1460, 1, 32, 64)
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
concat_one_year.shape: (1460, 5, 32, 64)
>>> 2018 <<<
geopotential_500: (1460, 1, 32, 64)
temperature_850: (1460, 1, 32, 64)
2m_temperature: (1460, 1, 32, 64)
10m_u_component_of_wind: (1460, 1, 32, 64)
<xarray.Dataset> Size: 12MB
Dimensions:                  (time: 1460, latitude: 64, longitude: 32)
Coordinates:
  * time                     (time) datetime64[

### Val set

In [28]:
var_list = ['geopotential_500', 'temperature_850', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind']

year_start, year_end = 2016, 2016

resolution = 5.625 

data_folder = '/mnt/data/sonia/weatherbench-5.625'

lat, long = 32, 64

In [29]:
select_merge_data(var_list, year_start, year_end, data_folder, resolution, lat, long)

>>> 2016 <<<
geopotential_500: (1464, 1, 32, 64)
temperature_850: (1464, 1, 32, 64)
2m_temperature: (1464, 1, 32, 64)
10m_u_component_of_wind: (1464, 1, 32, 64)
<xarray.Dataset> Size: 12MB
Dimensions:                  (time: 1464, latitude: 64, longitude: 32)
Coordinates:
  * time                     (time) datetime64[ns] 12kB 2016-01-01 ... 2016-1...
Dimensions without coordinates: latitude, longitude
Data variables:
    10m_v_component_of_wind  (time, latitude, longitude) float32 12MB ...
10m_v_component_of_wind: (1464, 1, 32, 64)
(1464, 1, 32, 64) 0
(1464, 1, 32, 64) 0
(1464, 1, 32, 64) 0
(1464, 1, 32, 64) 0
(1464, 1, 32, 64) 0
concat_one_year.shape: (1464, 5, 32, 64)
concat_years.shape: (1464, 5, 32, 64)
total time points: 1464
>>> saving data <<<
>>> saved data <<<


### Training set

In [30]:
var_list = ['geopotential_500', 'temperature_850', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind']

year_start, year_end = 2006, 2015

resolution = 5.625 

data_folder = '/mnt/data/sonia/weatherbench-5.625'

lat, long = 32, 64

In [31]:
select_merge_data(var_list, year_start, year_end, data_folder, resolution, lat, long)

>>> 2006 <<<
geopotential_500: (1460, 1, 32, 64)
temperature_850: (1460, 1, 32, 64)
2m_temperature: (1460, 1, 32, 64)
10m_u_component_of_wind: (1460, 1, 32, 64)
<xarray.Dataset> Size: 12MB
Dimensions:                  (time: 1460, latitude: 64, longitude: 32)
Coordinates:
  * time                     (time) datetime64[ns] 12kB 2006-01-01 ... 2006-1...
Dimensions without coordinates: latitude, longitude
Data variables:
    10m_v_component_of_wind  (time, latitude, longitude) float32 12MB ...
10m_v_component_of_wind: (1460, 1, 32, 64)
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
(1460, 1, 32, 64) 0
concat_one_year.shape: (1460, 5, 32, 64)
>>> 2007 <<<
geopotential_500: (1460, 1, 32, 64)
temperature_850: (1460, 1, 32, 64)
2m_temperature: (1460, 1, 32, 64)
10m_u_component_of_wind: (1460, 1, 32, 64)
<xarray.Dataset> Size: 12MB
Dimensions:                  (time: 1460, latitude: 64, longitude: 32)
Coordinates:
  * time                     (time) datetime64[